# Airline On-Time — cleaning & EDA

Quick pass over the Jan 2024 BTS sample before the Power BI model.
Goal: confirm null patterns, cancellation share, and that on-time % lines up with the dashboard (~75.8%).


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("../data")
OUT = Path("../python/outputs")
OUT.mkdir(parents=True, exist_ok=True)

flights = pd.read_csv(DATA / "fact_flights.csv")
airlines = pd.read_csv(DATA / "dim_airline.csv")
airports = pd.read_csv(DATA / "dim_airport.csv")
dates = pd.read_csv(DATA / "dim_date.csv")

print(flights.shape)
print(flights.columns.tolist()[:12], "...")
flights.head(3)


In [ ]:
# null / blank profile — cancelled rows drive most missing times
nulls = flights.isna().sum().sort_values(ascending=False)
print(nulls[nulls > 0].head(12))
print("blank TailNumber:", (flights["TailNumber"].fillna("").eq("")).sum())
print("dup FlightKey:", flights["FlightKey"].duplicated().sum())


In [ ]:
# operable = not cancelled, not diverted (same rule as dashboard on-time)
operable = flights[(flights["Cancelled"] == 0) & (flights["Diverted"] == 0)]
on_time_pct = operable["IsOnTimeArr"].mean() * 100
avg_arr = flights["ArrDelay"].clip(lower=0).mean()
cancel_pct = flights["Cancelled"].mean() * 100

print(f"rows: {len(flights):,}")
print(f"on-time % (operable): {on_time_pct:.1f}")
print(f"avg arr delay (floor 0): {avg_arr:.1f} min")
print(f"cancel %: {cancel_pct:.1f}")
print(f"delayed arr >=15: {flights['IsDelayedArr'].sum():,}")


In [ ]:
# carrier snapshot
m = flights.merge(airlines, on="AirlineKey")
rows = []
for (code, name), g in m.groupby(["AirlineCode", "AirlineName"]):
    op = g[(g.Cancelled == 0) & (g.Diverted == 0)]
    rows.append({
        "Airline": code,
        "Name": name,
        "Flights": len(g),
        "OnTime%": round(op["IsOnTimeArr"].mean() * 100, 1),
        "Cancel%": round(g["Cancelled"].mean() * 100, 1),
        "AvgArrDelay": round(g["ArrDelay"].clip(lower=0).mean(), 1),
    })
carrier = pd.DataFrame(rows).sort_values("Flights", ascending=False)
carrier


In [ ]:
# delay cause mix (minutes)
causes = {
    "Late aircraft": flights["LateAircraftDelay"].sum(),
    "Carrier": flights["CarrierDelay"].sum(),
    "NAS": flights["NASDelay"].sum(),
    "Weather": flights["WeatherDelay"].sum(),
    "Security": flights["SecurityDelay"].sum(),
}
cause_df = pd.Series(causes).sort_values(ascending=False)
print((cause_df / cause_df.sum() * 100).round(1))

fig, ax = plt.subplots(figsize=(7, 3.5))
cause_df.plot(kind="bar", ax=ax, color="#1F4E79")
ax.set_ylabel("Delay minutes")
ax.set_title("BTS delay minutes by cause — Jan 2024 sample")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
fig.savefig(OUT / "delay_cause_mix.png", dpi=120)
plt.show()


In [ ]:
# daily on-time
daily = (
    flights.merge(dates, on="DateKey")
    .groupby("Date")
    .apply(lambda g: pd.Series({
        "flights": len(g),
        "on_time_pct": g.loc[(g.Cancelled==0)&(g.Diverted==0), "IsOnTimeArr"].mean() * 100,
        "cancellations": g["Cancelled"].sum(),
    }), include_groups=False)
    .reset_index()
)
daily["Date"] = pd.to_datetime(daily["Date"])

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(daily["Date"], daily["on_time_pct"], marker="o", ms=3, color="#1F4E79")
ax.set_ylabel("On-time %")
ax.set_title("Daily on-time arrival %")
ax.set_ylim(40, 100)
plt.xticks(rotation=45)
plt.tight_layout()
fig.savefig(OUT / "daily_ontime.png", dpi=120)
plt.show()

print(daily.sort_values("on_time_pct").head(3)[["Date","on_time_pct","cancellations"]])
print(daily.sort_values("on_time_pct", ascending=False).head(1)[["Date","on_time_pct"]])


In [ ]:
# dep delay by time block — evening cascade
blk = (
    flights.groupby("DepTimeBlk")
    .agg(flights=("FlightKey","count"), dep_delay_pct=("IsDelayedDep", lambda s: 100*s.mean()))
    .reset_index()
    .sort_values("DepTimeBlk")
)
blk.tail(8)
